In [1]:
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------------------
# Rutas
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"

FEATURES_PATH = DATA_DIR / "04_player_tournament_features.csv"
EVALUATION_PATH = DATA_DIR / "pumas_player_evaluation_2025_2026.csv"
PCA_PATH = DATA_DIR / "pumas_pca_clustering_2025_2026.csv"

# ---------------------------------------------------------------------
# Carga
# ---------------------------------------------------------------------

features = pd.read_csv(FEATURES_PATH)
evaluation = pd.read_csv(EVALUATION_PATH)
pca = pd.read_csv(PCA_PATH)

print("=" * 80)
print("05 · PERFIL EXPLICATIVO DE JUGADORES")
print("=" * 80)

print("\nARCHIVOS CARGADOS")
print("-" * 80)
print(f"Features:   {features.shape}")
print(f"Evaluación: {evaluation.shape}")
print(f"PCA:        {pca.shape}")

# ---------------------------------------------------------------------
# Validación de columnas clave
# ---------------------------------------------------------------------

features_keys = [
    "season_name",
    "tournament_name",
    "player_id",
    "player_name",
    "position_group",
]

evaluation_keys = [
    "season_name",
    "tournament_name",
    "player_id",
    "player_name",
    "position_group",
]

pca_keys = [
    "tournament",
    "player_id",
    "player_name",
    "position_group",
]

for name, df, keys in [
    ("features", features, features_keys),
    ("evaluation", evaluation, evaluation_keys),
    ("pca", pca, pca_keys),
]:
    missing = [col for col in keys if col not in df.columns]

    if missing:
        raise ValueError(
            f"{name}: faltan columnas clave: {missing}"
        )

    print(f"\n{name.upper()}: columnas clave OK")

# ---------------------------------------------------------------------
# Duplicados de la unidad jugador + torneo
# ---------------------------------------------------------------------

features_duplicates = features.duplicated(
    subset=["season_name", "tournament_name", "player_id"]
).sum()

evaluation_duplicates = evaluation.duplicated(
    subset=["season_name", "tournament_name", "player_id"]
).sum()

pca_duplicates = pca.duplicated(
    subset=["tournament", "player_id"]
).sum()

print("\nDUPLICADOS EN CLAVES")
print("-" * 80)
print(f"Features:   {features_duplicates}")
print(f"Evaluación: {evaluation_duplicates}")
print(f"PCA:        {pca_duplicates}")

# ---------------------------------------------------------------------
# Cobertura de Pumas 2025/2026
# ---------------------------------------------------------------------

pumas_ids = set(evaluation["player_id"])

features_pumas = features[
    (features["season_name"] == "2025/2026")
    & (features["player_id"].isin(pumas_ids))
].copy()

print("\nCOBERTURA PUMAS 2025/2026")
print("-" * 80)

print(
    features_pumas.groupby("tournament_name")["player_id"]
    .nunique()
    .rename("features")
)

print("\nEvaluación:")
print(
    evaluation.groupby("tournament_name")["player_id"]
    .nunique()
    .rename("evaluation")
)

print("\nPCA:")
print(
    pca.groupby("tournament")["player_id"]
    .nunique()
    .rename("pca")
)

print("\nVALIDACIÓN INICIAL COMPLETADA")

05 · PERFIL EXPLICATIVO DE JUGADORES

ARCHIVOS CARGADOS
--------------------------------------------------------------------------------
Features:   (448, 90)
Evaluación: (39, 14)
PCA:        (36, 23)

FEATURES: columnas clave OK

EVALUATION: columnas clave OK

PCA: columnas clave OK

DUPLICADOS EN CLAVES
--------------------------------------------------------------------------------
Features:   2
Evaluación: 0
PCA:        0

COBERTURA PUMAS 2025/2026
--------------------------------------------------------------------------------
tournament_name
Apertura    20
Clausura    20
Name: features, dtype: int64

Evaluación:
tournament_name
Apertura    19
Clausura    20
Name: evaluation, dtype: int64

PCA:
tournament
Apertura    18
Clausura    18
Name: pca, dtype: int64

VALIDACIÓN INICIAL COMPLETADA


In [2]:
# ---------------------------------------------------------------------
# Diagnóstico de duplicados en features
# ---------------------------------------------------------------------

key_cols = [
    "season_name",
    "tournament_name",
    "player_id",
]

duplicate_mask = features.duplicated(
    subset=key_cols,
    keep=False
)

duplicates = (
    features.loc[duplicate_mask]
    .sort_values(key_cols + ["team_id"])
    .copy()
)

print("=" * 100)
print("DIAGNÓSTICO DE DUPLICADOS · 04_player_tournament_features")
print("=" * 100)

print(f"\nFilas involucradas: {len(duplicates)}")
print(
    "Claves jugador-torneo afectadas:",
    duplicates[key_cols].drop_duplicates().shape[0]
)

display_cols = [
    "season_name",
    "tournament_name",
    "player_id",
    "player_name",
    "team_id",
    "position_group",
    "appearances",
    "total_valid_minutes",
    "passes_per90",
    "key_passes_per90",
    "ball_recoveries_per90",
    "interceptions_per90",
    "provider_rating_tournament",
]

display(duplicates[display_cols])

print("\n" + "=" * 100)
print("DIFERENCIAS ENTRE FILAS DUPLICADAS")
print("=" * 100)

for key, group in duplicates.groupby(key_cols, sort=False):

    season, tournament, player_id = key
    player_name = group["player_name"].iloc[0]

    print(
        f"\n{player_name} · player_id={player_id} · "
        f"{season} · {tournament}"
    )
    print("-" * 100)

    varying = []

    for col in features.columns:
        values = group[col].drop_duplicates()

        if len(values) > 1:
            varying.append(col)

    print(f"Columnas con valores diferentes: {len(varying)}")

    for col in varying:
        print(f"  - {col}: {group[col].tolist()}")

DIAGNÓSTICO DE DUPLICADOS · 04_player_tournament_features

Filas involucradas: 4
Claves jugador-torneo afectadas: 2


,season_name,tournament_name,player_id,player_name,team_id,position_group,appearances,total_valid_minutes,passes_per90,key_passes_per90,ball_recoveries_per90,interceptions_per90,provider_rating_tournament
293,2025/2026,Clausura,142,Carlos Uriel Antuna Romero,1,FW,9,476,24.768908,0.945378,3.025210,0.378151,7.068508
294,2025/2026,Clausura,142,Carlos Uriel Antuna Romero,5,FW,1,46,48.913043,1.956522,0.000000,1.956522,6.800000
304,2025/2026,Clausura,162,Nicolás Alejandro Ibáñez,5,FW,1,57,12.631579,1.578947,1.578947,0.000000,6.700000
305,2025/2026,Clausura,162,Nicolás Alejandro Ibáñez,8,FW,1,71,17.746479,0.000000,0.000000,0.000000,7.000000



DIFERENCIAS ENTRE FILAS DUPLICADAS

Carlos Uriel Antuna Romero · player_id=142 · 2025/2026 · Clausura
----------------------------------------------------------------------------------------------------
Columnas con valores diferentes: 74
  - team_id: [1, 5]
  - appearances: [9, 1]
  - total_valid_minutes: [476, 46]
  - total_accurate_crosses: [3, 0]
  - total_accurate_passes: [110, 24]
  - total_aerials: [13.0, nan]
  - total_aerials_lost: [6.0, nan]
  - total_aerials_won: [7.0, nan]
  - total_assists: [3, 0]
  - total_backward_passes: [41, 7]
  - total_ball_recoveries: [16, 0]
  - total_clearances: [4, 0]
  - total_dispossessed: [5.0, 0.0]
  - total_dribble_attempts: [4, 1]
  - total_dribbled_past: [2, 0]
  - total_duels_lost: [13, 1]
  - total_duels_won: [15, 1]
  - total_fouls: [2, 0]
  - total_fouls_drawn: [2, 1]
  - total_goals: [2, 0]
  - total_interceptions: [2, 1]
  - total_key_passes: [5, 1]
  - total_long_balls: [8, 0]
  - total_long_balls_won: [6, 0]
  - total_passes: [131

In [3]:
# ---------------------------------------------------------------------
# Jugadores que participaron con más de un equipo en el mismo torneo
# ---------------------------------------------------------------------

multi_team = (
    features
    .groupby(
        [
            "season_name",
            "tournament_name",
            "player_id",
            "player_name",
            "position_group",
        ],
        as_index=False
    )
    .agg(
        teams=("team_id", "nunique"),
        team_ids=("team_id", lambda x: sorted(set(x))),
        appearances=("appearances", "sum"),
        total_valid_minutes=("total_valid_minutes", "sum"),
    )
)

multi_team = (
    multi_team[multi_team["teams"] > 1]
    .sort_values(
        ["season_name", "tournament_name", "player_name"]
    )
    .reset_index(drop=True)
)

print("=" * 100)
print("JUGADORES MULTI-EQUIPO EN EL MISMO TORNEO")
print("=" * 100)

print(f"\nCasos encontrados: {len(multi_team)}")

if len(multi_team):
    display(multi_team)
else:
    print("No se encontraron casos.")

print("\n" + "=" * 100)
print("CASOS QUE SUPERARÍAN 450 MINUTOS TRAS AGREGAR EQUIPOS")
print("=" * 100)

eligible_multi_team = multi_team[
    multi_team["total_valid_minutes"] >= 450
].copy()

if len(eligible_multi_team):
    display(eligible_multi_team)
else:
    print("Ninguno.")

JUGADORES MULTI-EQUIPO EN EL MISMO TORNEO

Casos encontrados: 2


,season_name,tournament_name,player_id,player_name,position_group,teams,team_ids,appearances,total_valid_minutes
0,2025/2026,Clausura,142,Carlos Uriel Antuna Romero,FW,2,"[1, 5]",10,522
1,2025/2026,Clausura,162,Nicolás Alejandro Ibáñez,FW,2,"[5, 8]",2,128



CASOS QUE SUPERARÍAN 450 MINUTOS TRAS AGREGAR EQUIPOS


,season_name,tournament_name,player_id,player_name,position_group,teams,team_ids,appearances,total_valid_minutes
0,2025/2026,Clausura,142,Carlos Uriel Antuna Romero,FW,2,"[1, 5]",10,522


In [8]:
# ---------------------------------------------------------------------
# Consolidación a nivel jugador + temporada + torneo
# ---------------------------------------------------------------------

import numpy as np

group_cols = [
    "season_id",
    "season_name",
    "tournament_name",
    "player_id",
    "player_name",
    "position_group",
]

# Columnas acumulativas que pueden sumarse entre equipos
total_cols = [
    col for col in features.columns
    if col.startswith("total_")
]

# appearances también es acumulativa
agg_dict = {
    "appearances": "sum",
    "total_valid_minutes": "sum",
}

for col in total_cols:
    agg_dict[col] = lambda x: x.sum(min_count=1)

# Consolidamos cualquier caso multi-equipo
player_tournament = (
    features
    .groupby(group_cols, as_index=False, dropna=False)
    .agg(agg_dict)
)

# ---------------------------------------------------------------------
# Recalcular métricas por 90 desde los totales consolidados
# ---------------------------------------------------------------------

per90_map = {
    "passes_per90": "total_passes",
    "accurate_passes_per90": "total_accurate_passes",
    "key_passes_per90": "total_key_passes",
    "passes_in_final_third_per90": "total_passes_in_final_third",
    "backward_passes_per90": "total_backward_passes",
    "touches_per90": "total_touches",
    "possession_lost_per90": "total_possession_lost",
    "dispossessed_per90": "total_dispossessed",
    "goals_per90": "total_goals",
    "assists_per90": "total_assists",
    "shots_per90": "total_shots",
    "shots_on_target_per90": "total_shots_on_target",
    "dribble_attempts_per90": "total_dribble_attempts",
    "successful_dribbles_per90": "total_successful_dribbles",
    "total_crosses_per90": "total_total_crosses",
    "accurate_crosses_per90": "total_accurate_crosses",
    "tackles_per90": "total_tackles",
    "interceptions_per90": "total_interceptions",
    "ball_recoveries_per90": "total_ball_recoveries",
    "clearances_per90": "total_clearances",
    "dribbled_past_per90": "total_dribbled_past",
    "total_duels_per90": "total_total_duels",
    "duels_won_per90": "total_duels_won",
    "duels_lost_per90": "total_duels_lost",
    "aerials_per90": "total_aerials",
    "aerials_won_per90": "total_aerials_won",
    "aerials_lost_per90": "total_aerials_lost",
    "fouls_per90": "total_fouls",
    "fouls_drawn_per90": "total_fouls_drawn",
    "yellow_cards_per90": "total_yellow_cards",
    "red_cards_per90": "total_red_cards",
    "long_balls_per90": "total_long_balls",
    "long_balls_won_per90": "total_long_balls_won",
    "saves_per90": "total_saves",
    "saves_inside_box_per90": "total_saves_inside_box",
    "goals_conceded_per90": "total_goals_conceded",
}

minutes = player_tournament["total_valid_minutes"]

for new_col, total_col in per90_map.items():
    player_tournament[new_col] = np.where(
        minutes > 0,
        player_tournament[total_col] / minutes * 90,
        np.nan,
    )

# ---------------------------------------------------------------------
# Recalcular tasas desde numerador / denominador
# ---------------------------------------------------------------------

def safe_rate(numerator, denominator):
    return np.where(
        denominator > 0,
        numerator / denominator,
        np.nan,
    )

player_tournament["pass_accuracy"] = safe_rate(
    player_tournament["total_accurate_passes"],
    player_tournament["total_passes"],
)

player_tournament["cross_accuracy"] = safe_rate(
    player_tournament["total_accurate_crosses"],
    player_tournament["total_total_crosses"],
)

player_tournament["dribble_success"] = safe_rate(
    player_tournament["total_successful_dribbles"],
    player_tournament["total_dribble_attempts"],
)

player_tournament["duel_success"] = safe_rate(
    player_tournament["total_duels_won"],
    player_tournament["total_total_duels"],
)

player_tournament["aerial_success"] = safe_rate(
    player_tournament["total_aerials_won"],
    player_tournament["total_aerials"],
)

player_tournament["long_ball_success"] = safe_rate(
    player_tournament["total_long_balls_won"],
    player_tournament["total_long_balls"],
)

# Para porteros: tiros a puerta enfrentados = paradas + goles recibidos
player_tournament["shots_faced_on_target"] = (
    player_tournament["total_saves"]
    + player_tournament["total_goals_conceded"]
)

player_tournament["save_rate"] = safe_rate(
    player_tournament["total_saves"],
    player_tournament["shots_faced_on_target"],
)

# ---------------------------------------------------------------------
# Validaciones
# ---------------------------------------------------------------------

duplicates_after = player_tournament.duplicated(
    subset=["season_name", "tournament_name", "player_id"]
).sum()

print("=" * 90)
print("BASE CONSOLIDADA · JUGADOR + TORNEO")
print("=" * 90)

print(f"\nFilas originales:    {len(features)}")
print(f"Filas consolidadas:  {len(player_tournament)}")
print(f"Duplicados restantes: {duplicates_after}")

print("\nANTUNA · CLAUSURA 2025/2026")
print("-" * 90)

antuna = player_tournament[
    (player_tournament["season_name"] == "2025/2026")
    & (player_tournament["tournament_name"] == "Clausura")
    & (player_tournament["player_id"] == 142)
]

display(
    antuna[
        [
            "player_name",
            "appearances",
            "total_valid_minutes",
            "total_goals",
            "total_assists",
            "goals_per90",
            "assists_per90",
            "passes_per90",
            "key_passes_per90",
        ]
    ]
)

print("\nBENCHMARK >= 90 MINUTOS")
print("-" * 90)

benchmark = player_tournament[
    player_tournament["total_valid_minutes"] >= 90
].copy()

summary = (
    benchmark
    .groupby(
        ["season_name", "tournament_name", "position_group"]
    )
    .size()
    .rename("jugadores")
    .reset_index()
)

display(summary)

print(f"Total jugador-torneo elegibles: {len(benchmark)}")

BASE CONSOLIDADA · JUGADOR + TORNEO

Filas originales:    448
Filas consolidadas:  446
Duplicados restantes: 0

ANTUNA · CLAUSURA 2025/2026
------------------------------------------------------------------------------------------


,player_name,appearances,total_valid_minutes,total_goals,total_assists,goals_per90,assists_per90,passes_per90,key_passes_per90
293,Carlos Uriel Antuna Romero,10,522,2,3,0.344828,0.517241,26.896552,1.034483



BENCHMARK >= 90 MINUTOS
------------------------------------------------------------------------------------------


,season_name,tournament_name,position_group,jugadores
0,2025/2026,Apertura,DF,69
1,2025/2026,Apertura,FW,18
2,2025/2026,Apertura,GK,17
3,2025/2026,Apertura,MF,36
4,2025/2026,Clausura,DF,60
5,2025/2026,Clausura,FW,24
6,2025/2026,Clausura,GK,18
7,2025/2026,Clausura,MF,33


Total jugador-torneo elegibles: 275


In [9]:
# ---------------------------------------------------------------------
# Configuración de métricas explicativas por posición
# ---------------------------------------------------------------------

MIN_BENCHMARK_MINUTES = 90

metric_config = {
    "GK": {
        "saves_per90": {
            "label": "Paradas",
            "direction": "positive",
            "format": "per90",
        },
        "saves_inside_box_per90": {
            "label": "Paradas dentro del área",
            "direction": "positive",
            "format": "per90",
        },
        "goals_conceded_per90": {
            "label": "Goles recibidos",
            "direction": "inverse",
            "format": "per90",
        },
        "save_rate": {
            "label": "Porcentaje de paradas",
            "direction": "positive",
            "format": "percentage",
        },
        "pass_accuracy": {
            "label": "Precisión de pase",
            "direction": "descriptive",
            "format": "percentage",
        },
    },

    "DF": {
        "tackles_per90": {
            "label": "Entradas",
            "direction": "positive",
            "format": "per90",
        },
        "interceptions_per90": {
            "label": "Intercepciones",
            "direction": "positive",
            "format": "per90",
        },
        "ball_recoveries_per90": {
            "label": "Recuperaciones",
            "direction": "positive",
            "format": "per90",
        },
        "clearances_per90": {
            "label": "Despejes",
            "direction": "descriptive",
            "format": "per90",
        },
        "duels_won_per90": {
            "label": "Duelos ganados",
            "direction": "positive",
            "format": "per90",
        },
        "pass_accuracy": {
            "label": "Precisión de pase",
            "direction": "descriptive",
            "format": "percentage",
        },
    },

    "MF": {
        "key_passes_per90": {
            "label": "Pases clave",
            "direction": "positive",
            "format": "per90",
        },
        "passes_in_final_third_per90": {
            "label": "Pases en último tercio",
            "direction": "descriptive",
            "format": "per90",
        },
        "successful_dribbles_per90": {
            "label": "Regates exitosos",
            "direction": "positive",
            "format": "per90",
        },
        "ball_recoveries_per90": {
            "label": "Recuperaciones",
            "direction": "positive",
            "format": "per90",
        },
        "duels_won_per90": {
            "label": "Duelos ganados",
            "direction": "positive",
            "format": "per90",
        },
        "pass_accuracy": {
            "label": "Precisión de pase",
            "direction": "descriptive",
            "format": "percentage",
        },
    },

    "FW": {
        "goals_per90": {
            "label": "Goles",
            "direction": "positive",
            "format": "per90",
        },
        "assists_per90": {
            "label": "Asistencias",
            "direction": "positive",
            "format": "per90",
        },
        "shots_per90": {
            "label": "Tiros",
            "direction": "descriptive",
            "format": "per90",
        },
        "shots_on_target_per90": {
            "label": "Tiros a puerta",
            "direction": "positive",
            "format": "per90",
        },
        "successful_dribbles_per90": {
            "label": "Regates exitosos",
            "direction": "positive",
            "format": "per90",
        },
        "key_passes_per90": {
            "label": "Pases clave",
            "direction": "positive",
            "format": "per90",
        },
    },
}

# ---------------------------------------------------------------------
# Benchmark elegible
# ---------------------------------------------------------------------

benchmark = player_tournament[
    player_tournament["total_valid_minutes"] >= MIN_BENCHMARK_MINUTES
].copy()

# ---------------------------------------------------------------------
# Construcción de referencias
# ---------------------------------------------------------------------

reference_rows = []

for (season, tournament, position), group in benchmark.groupby(
    ["season_name", "tournament_name", "position_group"]
):
    if position not in metric_config:
        continue

    for metric, config in metric_config[position].items():
        values = group[metric].dropna()

        if values.empty:
            continue

        reference_rows.append(
            {
                "season_name": season,
                "tournament_name": tournament,
                "position_group": position,
                "metric": metric,
                "metric_label": config["label"],
                "direction": config["direction"],
                "display_format": config["format"],
                "reference_mean": values.mean(),
                "reference_median": values.median(),
                "benchmark_n": len(values),
            }
        )

references = pd.DataFrame(reference_rows)

# ---------------------------------------------------------------------
# Validación
# ---------------------------------------------------------------------

print("=" * 100)
print("REFERENCIAS POR POSICIÓN Y TORNEO")
print(f"Benchmark mínimo: {MIN_BENCHMARK_MINUTES} minutos")
print("=" * 100)

print(f"\nFilas de referencia generadas: {len(references)}")

print("\nTAMAÑO DEL BENCHMARK")
display(
    references[
        [
            "season_name",
            "tournament_name",
            "position_group",
            "benchmark_n",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["season_name", "tournament_name", "position_group"]
    )
)

print("\nEJEMPLO · DEFENSAS · APERTURA 2025/2026")
display(
    references[
        (references["season_name"] == "2025/2026")
        & (references["tournament_name"] == "Apertura")
        & (references["position_group"] == "DF")
    ][
        [
            "metric_label",
            "direction",
            "reference_mean",
            "reference_median",
            "benchmark_n",
        ]
    ]
)

REFERENCIAS POR POSICIÓN Y TORNEO
Benchmark mínimo: 90 minutos

Filas de referencia generadas: 46

TAMAÑO DEL BENCHMARK


,season_name,tournament_name,position_group,benchmark_n
0,2025/2026,Apertura,DF,69
6,2025/2026,Apertura,FW,18
12,2025/2026,Apertura,GK,17
15,2025/2026,Apertura,GK,16
17,2025/2026,Apertura,MF,36
23,2025/2026,Clausura,DF,60
29,2025/2026,Clausura,FW,24
35,2025/2026,Clausura,GK,18
40,2025/2026,Clausura,MF,33



EJEMPLO · DEFENSAS · APERTURA 2025/2026


,metric_label,direction,reference_mean,reference_median,benchmark_n
0,Entradas,positive,1.576458,1.000000,69
1,Intercepciones,positive,0.831792,1.000000,69
2,Recuperaciones,positive,3.084025,3.000000,69
3,Despejes,descriptive,3.980433,4.000000,69
4,Duelos ganados,positive,3.978184,3.000000,69
5,Precisión de pase,descriptive,0.851670,0.866667,69


In [10]:
# ---------------------------------------------------------------------
# Comparación de un jugador contra la referencia de su posición
# ---------------------------------------------------------------------

def player_metric_comparison(player_id, tournament):
    # Identificar evaluación del jugador
    ev = evaluation[
        (evaluation["player_id"] == player_id)
        & (evaluation["tournament_name"] == tournament)
    ]

    if ev.empty:
        raise ValueError(
            f"No existe evaluación para player_id={player_id}, "
            f"torneo={tournament}"
        )

    ev = ev.iloc[0]

    season = ev["season_name"]
    position = ev["position_group"]

    # Buscar features consolidadas
    player = player_tournament[
        (player_tournament["season_name"] == season)
        & (player_tournament["tournament_name"] == tournament)
        & (player_tournament["player_id"] == player_id)
    ]

    if player.empty:
        raise ValueError("No se encontraron features del jugador.")

    player = player.iloc[0]

    rows = []

    for metric, config in metric_config[position].items():

        ref = references[
            (references["season_name"] == season)
            & (references["tournament_name"] == tournament)
            & (references["position_group"] == position)
            & (references["metric"] == metric)
        ]

        if ref.empty:
            continue

        ref = ref.iloc[0]

        value = player[metric]
        reference = ref["reference_mean"]

        if pd.isna(value) or pd.isna(reference):
            continue

        difference = value - reference

        # Diferencia relativa solo cuando la referencia permite
        # una comparación porcentual estable.
        if abs(reference) > 0.10:
            relative_difference = difference / abs(reference)
        else:
            relative_difference = np.nan

        # Score de comparación para ordenar métricas favorables.
        # Las descriptivas no reciben interpretación de calidad.
        if config["direction"] == "positive":
            comparison_score = relative_difference

        elif config["direction"] == "inverse":
            comparison_score = (
                -relative_difference
                if not pd.isna(relative_difference)
                else np.nan
            )

        else:
            comparison_score = np.nan

        rows.append(
            {
                "metric": metric,
                "metric_label": config["label"],
                "direction": config["direction"],
                "display_format": config["format"],
                "player_value": value,
                "reference_value": reference,
                "absolute_difference": difference,
                "relative_difference": relative_difference,
                "comparison_score": comparison_score,
                "benchmark_n": ref["benchmark_n"],
            }
        )

    result = pd.DataFrame(rows)

    return ev, result


# ---------------------------------------------------------------------
# Prueba: Rubén Duarte
# ---------------------------------------------------------------------

for tournament in ["Apertura", "Clausura"]:

    ev, comparison = player_metric_comparison(
        player_id=37,
        tournament=tournament,
    )

    print("\n" + "=" * 100)
    print(
        f"{ev['player_name'].upper()} · "
        f"{tournament.upper()} {ev['season_name']}"
    )
    print("=" * 100)

    print(
        f"Posición: {ev['position_group']} | "
        f"Minutos: {ev['total_valid_minutes']:.0f} | "
        f"PPS: {ev['pps_v1']:.2f} | "
        f"IGR: {ev['igr_v1']:.2f}"
    )

    view = comparison.copy()

    view["difference_pct"] = (
        view["relative_difference"] * 100
    )

    display(
        view[
            [
                "metric_label",
                "direction",
                "player_value",
                "reference_value",
                "difference_pct",
                "benchmark_n",
            ]
        ].round(
            {
                "player_value": 3,
                "reference_value": 3,
                "difference_pct": 1,
            }
        )
    )


RUBÉN DUARTE SÁNCHEZ · APERTURA 2025/2026
Posición: DF | Minutos: 1260 | PPS: 62.34 | IGR: 78.30


,metric_label,direction,player_value,reference_value,difference_pct,benchmark_n
0,Entradas,positive,1.643,1.576,4.2,69
1,Intercepciones,positive,1.071,0.832,28.8,69
2,Recuperaciones,positive,4.000,3.084,29.7,69
3,Despejes,descriptive,4.571,3.980,14.8,69
4,Duelos ganados,positive,4.214,3.978,5.9,69
5,Precisión de pase,descriptive,0.877,0.852,2.9,69



RUBÉN DUARTE SÁNCHEZ · CLAUSURA 2025/2026
Posición: DF | Minutos: 755 | PPS: 45.20 | IGR: 39.56


,metric_label,direction,player_value,reference_value,difference_pct,benchmark_n
0,Entradas,positive,0.715,1.298,-44.9,60
1,Intercepciones,positive,1.311,1.060,23.6,60
2,Recuperaciones,positive,2.861,2.887,-0.9,60
3,Despejes,descriptive,5.126,4.866,5.3,60
4,Duelos ganados,positive,2.980,4.178,-28.7,60
5,Precisión de pase,descriptive,0.891,0.853,4.5,60


In [12]:
# ---------------------------------------------------------------------
# Clasificación interpretable de todas las métricas
# ---------------------------------------------------------------------

HIGHLIGHT_THRESHOLD = 0.10


def classify_metric(row):
    direction = row["direction"]
    relative_difference = row["relative_difference"]
    comparison_score = row["comparison_score"]

    if pd.isna(relative_difference):
        return "Sin comparación"

    # Métricas positivas/inversas:
    # la clasificación tiene interpretación de rendimiento.
    if direction in ["positive", "inverse"]:

        if comparison_score >= HIGHLIGHT_THRESHOLD:
            return "Destaca"

        if comparison_score <= -HIGHLIGHT_THRESHOLD:
            return "Por debajo"

        return "En línea"

    # Métricas descriptivas:
    # clasificamos su posición respecto a la referencia,
    # pero SIN interpretar "más" como mejor rendimiento.
    if relative_difference >= HIGHLIGHT_THRESHOLD:
        return "Destaca"

    if relative_difference <= -HIGHLIGHT_THRESHOLD:
        return "Por debajo"

    return "En línea"


# ---------------------------------------------------------------------
# Validación: Rubén Duarte
# ---------------------------------------------------------------------

for tournament in ["Apertura", "Clausura"]:

    ev, comparison = player_metric_comparison(
        player_id=37,
        tournament=tournament,
    )

    comparison["classification"] = comparison.apply(
        classify_metric,
        axis=1,
    )

    comparison["difference_pct"] = (
        comparison["relative_difference"] * 100
    )

    print("\n" + "=" * 100)
    print(
        f"{ev['player_name'].upper()} · "
        f"{tournament.upper()} {ev['season_name']}"
    )
    print("=" * 100)

    display(
        comparison[
            [
                "metric_label",
                "direction",
                "player_value",
                "reference_value",
                "difference_pct",
                "classification",
            ]
        ].round(
            {
                "player_value": 3,
                "reference_value": 3,
                "difference_pct": 1,
            }
        )
    )


RUBÉN DUARTE SÁNCHEZ · APERTURA 2025/2026


,metric_label,direction,player_value,reference_value,difference_pct,classification
0,Entradas,positive,1.643,1.576,4.2,En línea
1,Intercepciones,positive,1.071,0.832,28.8,Destaca
2,Recuperaciones,positive,4.000,3.084,29.7,Destaca
3,Despejes,descriptive,4.571,3.980,14.8,Destaca
4,Duelos ganados,positive,4.214,3.978,5.9,En línea
5,Precisión de pase,descriptive,0.877,0.852,2.9,En línea



RUBÉN DUARTE SÁNCHEZ · CLAUSURA 2025/2026


,metric_label,direction,player_value,reference_value,difference_pct,classification
0,Entradas,positive,0.715,1.298,-44.9,Por debajo
1,Intercepciones,positive,1.311,1.060,23.6,Destaca
2,Recuperaciones,positive,2.861,2.887,-0.9,En línea
3,Despejes,descriptive,5.126,4.866,5.3,En línea
4,Duelos ganados,positive,2.980,4.178,-28.7,Por debajo
5,Precisión de pase,descriptive,0.891,0.853,4.5,En línea


In [13]:
# ---------------------------------------------------------------------
# Construcción y validación de perfiles para todos los jugadores Pumas
# ---------------------------------------------------------------------

profile_rows = []

for _, ev in evaluation.iterrows():

    player_id = ev["player_id"]
    tournament = ev["tournament_name"]

    try:
        _, comparison = player_metric_comparison(
            player_id=player_id,
            tournament=tournament,
        )
    except ValueError as error:
        print(
            f"Sin perfil: {ev['player_name']} · "
            f"{tournament} · {error}"
        )
        continue

    comparison["classification"] = comparison.apply(
        classify_metric,
        axis=1,
    )

    for _, metric_row in comparison.iterrows():

        profile_rows.append(
            {
                "season_name": ev["season_name"],
                "tournament_name": tournament,
                "player_id": player_id,
                "player_name": ev["player_name"],
                "position_group": ev["position_group"],
                "appearances": ev["appearances"],
                "total_valid_minutes": ev["total_valid_minutes"],
                "pps_v1": ev["pps_v1"],
                "igr_v1": ev["igr_v1"],

                "metric": metric_row["metric"],
                "metric_label": metric_row["metric_label"],
                "direction": metric_row["direction"],
                "display_format": metric_row["display_format"],

                "player_value": metric_row["player_value"],
                "reference_value": metric_row["reference_value"],
                "absolute_difference": metric_row["absolute_difference"],
                "relative_difference": metric_row["relative_difference"],
                "comparison_score": metric_row["comparison_score"],

                "classification": metric_row["classification"],
                "benchmark_n": metric_row["benchmark_n"],
            }
        )

profiles = pd.DataFrame(profile_rows)

print("=" * 100)
print("PERFILES EXPLICATIVOS · PUMAS 2025/2026")
print("=" * 100)

n_profiles = profiles[
    ["tournament_name", "player_id"]
].drop_duplicates().shape[0]

print(f"\nJugador-torneo evaluados:    {len(evaluation)}")
print(f"Jugador-torneo con perfil:   {n_profiles}")
print(f"Métricas generadas:          {len(profiles)}")

print("\n" + "=" * 100)
print("DISTRIBUCIÓN DE CLASIFICACIONES")
print("=" * 100)

classification_summary = (
    profiles
    .groupby(["position_group", "classification"])
    .size()
    .unstack(fill_value=0)
)

display(classification_summary)

print("\n" + "=" * 100)
print("EJEMPLO CON MAYOR IGR POR POSICIÓN")
print("=" * 100)

for position in ["GK", "DF", "MF", "FW"]:

    subset = profiles[
        profiles["position_group"] == position
    ]

    if subset.empty:
        continue

    example_key = (
        subset[
            [
                "tournament_name",
                "player_id",
                "player_name",
                "igr_v1",
            ]
        ]
        .drop_duplicates()
        .sort_values("igr_v1", ascending=False)
        .iloc[0]
    )

    example = subset[
        (subset["tournament_name"] == example_key["tournament_name"])
        & (subset["player_id"] == example_key["player_id"])
    ].copy()

    example["difference_pct"] = (
        example["relative_difference"] * 100
    )

    print("\n" + "-" * 100)
    print(
        f"{position} · "
        f"{example_key['player_name']} · "
        f"{example_key['tournament_name']} · "
        f"IGR {example_key['igr_v1']:.2f}"
    )
    print("-" * 100)

    display(
        example[
            [
                "metric_label",
                "player_value",
                "reference_value",
                "difference_pct",
                "classification",
                "benchmark_n",
            ]
        ].round(
            {
                "player_value": 3,
                "reference_value": 3,
                "difference_pct": 1,
            }
        )
    )

PERFILES EXPLICATIVOS · PUMAS 2025/2026

Jugador-torneo evaluados:    39
Jugador-torneo con perfil:   39
Métricas generadas:          231

DISTRIBUCIÓN DE CLASIFICACIONES


classification,Destaca,En línea,Por debajo
position_group,,,
DF,24,27,39
FW,25,13,34
GK,8,3,4
MF,22,18,14



EJEMPLO CON MAYOR IGR POR POSICIÓN

----------------------------------------------------------------------------------------------------
GK · Keylor Navas Gamboa · Clausura · IGR 75.40
----------------------------------------------------------------------------------------------------


,metric_label,player_value,reference_value,difference_pct,classification,benchmark_n
125,Paradas,4.500,2.583,74.2,Destaca,18
126,Paradas dentro del área,2.562,1.476,73.6,Destaca,18
127,Goles recibidos,0.938,1.830,-48.8,Destaca,18
128,Porcentaje de paradas,0.828,0.538,53.7,Destaca,18
129,Precisión de pase,0.716,0.803,-10.7,Por debajo,18



----------------------------------------------------------------------------------------------------
DF · Rubén Duarte Sánchez · Apertura · IGR 78.30
----------------------------------------------------------------------------------------------------


,metric_label,player_value,reference_value,difference_pct,classification,benchmark_n
0,Entradas,1.643,1.576,4.2,En línea,69
1,Intercepciones,1.071,0.832,28.8,Destaca,69
2,Recuperaciones,4.000,3.084,29.7,Destaca,69
3,Despejes,4.571,3.980,14.8,Destaca,69
4,Duelos ganados,4.214,3.978,5.9,En línea,69
5,Precisión de pase,0.877,0.852,2.9,En línea,69



----------------------------------------------------------------------------------------------------
MF · Adalberto Eliécer Carrasquilla Alcázar · Clausura · IGR 81.76
----------------------------------------------------------------------------------------------------


,metric_label,player_value,reference_value,difference_pct,classification,benchmark_n
113,Pases clave,1.458,1.531,-4.8,En línea,33
114,Pases en último tercio,7.673,7.359,4.3,En línea,33
115,Regates exitosos,1.151,0.523,120.2,Destaca,33
116,Recuperaciones,5.448,4.494,21.2,Destaca,33
117,Duelos ganados,4.143,3.865,7.2,En línea,33
118,Precisión de pase,0.825,0.842,-2.0,En línea,33



----------------------------------------------------------------------------------------------------
FW · Jordán Carrillo Rodríguez · Clausura · IGR 72.25
----------------------------------------------------------------------------------------------------


,metric_label,player_value,reference_value,difference_pct,classification,benchmark_n
130,Goles,0.279,0.272,2.8,En línea,24
131,Asistencias,0.186,0.205,-9.0,En línea,24
132,Tiros,2.141,2.824,-24.2,Por debajo,24
133,Tiros a puerta,0.745,0.998,-25.4,Por debajo,24
134,Regates exitosos,1.303,0.804,62.1,Destaca,24
135,Pases clave,1.675,1.520,10.2,Destaca,24


## Metodología de comparación del perfil individual

Las métricas del perfil individual se comparan contra una referencia construida con
jugadores de la misma temporada, torneo y grupo de posición.

### Benchmark

Para formar parte del benchmark, un jugador debe acumular al menos **90 minutos
válidos** en el torneo.

Cuando un jugador registra participación con más de un equipo dentro del mismo torneo,
sus estadísticas se consolidan primero a nivel **jugador + temporada + torneo**.
Los totales se suman y las métricas por 90 minutos y tasas se recalculan utilizando
los minutos consolidados.

La referencia utilizada en el producto es la **media del grupo de posición**.

### Clasificación

Para facilitar la interpretación en la interfaz, cada métrica se compara con su
referencia mediante una diferencia relativa.

Se utilizan tres categorías:

- **Destaca:** diferencia favorable o superior a la referencia de al menos 10%.
- **En línea:** diferencia dentro del intervalo de ±10% respecto a la referencia.
- **Por debajo:** diferencia desfavorable o inferior a la referencia de al menos 10%.

En métricas de dirección inversa, como goles recibidos por 90 minutos, una cifra menor
que la referencia se interpreta favorablemente.

### Métricas descriptivas

Algunas métricas representan características del juego y no calidad de rendimiento
por sí mismas. Ejemplos son volumen de tiros, despejes, pases en último tercio o
precisión de pase.

Estas métricas utilizan las mismas tres categorías para mantener una presentación
uniforme, pero una clasificación **Destaca** significa que el valor se encuentra
claramente por encima de la referencia y no necesariamente que el jugador sea mejor
por producir una cifra mayor.

### Alcance

Estas comparaciones complementan PPS e IGR:

- **PPS / IGR:** resumen cómo rindió el jugador.
- **Perfil individual:** muestra qué métricas ayudan a caracterizar y explicar ese rendimiento.
- **PCA / clustering:** describe el perfil estadístico general del jugador.

Las comparaciones de Apertura y Clausura se mantienen independientes.

In [15]:
# ---------------------------------------------------------------------
# Paso 46 · Integración PPS / IGR + métricas explicativas
# ---------------------------------------------------------------------

evaluation_cols = [
    "season_name",
    "tournament_name",
    "player_id",
    "player_name",
    "position_group",
    "appearances",
    "total_valid_minutes",
    "pps_v1",
    "pps_position_rank",
    "benchmark_pps_pct",
    "igr_reliability",
    "igr_v1",
    "igr_rank",
    "provider_rating_tournament",
]

metric_cols = [
    "season_name",
    "tournament_name",
    "player_id",
    "metric",
    "metric_label",
    "direction",
    "display_format",
    "player_value",
    "reference_value",
    "absolute_difference",
    "relative_difference",
    "comparison_score",
    "classification",
    "benchmark_n",
]

# Una fila por jugador + torneo
evaluation_base = evaluation[evaluation_cols].copy()

# Una fila por jugador + torneo + métrica
metrics_base = profiles[metric_cols].copy()

# ---------------------------------------------------------------------
# Unión
# ---------------------------------------------------------------------

player_performance = evaluation_base.merge(
    metrics_base,
    on=[
        "season_name",
        "tournament_name",
        "player_id",
    ],
    how="left",
    validate="one_to_many",
)

# Diferencia porcentual preparada para presentación/API.
player_performance["difference_pct"] = (
    player_performance["relative_difference"] * 100
)

# ---------------------------------------------------------------------
# Validaciones estructurales
# ---------------------------------------------------------------------

n_player_tournaments = (
    player_performance[
        ["season_name", "tournament_name", "player_id"]
    ]
    .drop_duplicates()
    .shape[0]
)

duplicate_metrics = (
    player_performance
    .duplicated(
        subset=[
            "season_name",
            "tournament_name",
            "player_id",
            "metric",
        ]
    )
    .sum()
)

print("=" * 100)
print("RENDIMIENTO INDIVIDUAL · PPS / IGR + MÉTRICAS EXPLICATIVAS")
print("=" * 100)

print(f"\nJugador-torneo esperados:       {len(evaluation)}")
print(f"Jugador-torneo integrados:      {n_player_tournaments}")
print(f"Métricas explicativas:          {len(player_performance)}")
print(f"Duplicados jugador-métrica:     {duplicate_metrics}")
print(
    "Filas sin métrica:             "
    f"{player_performance['metric'].isna().sum()}"
)

print("\nCLASIFICACIONES")
print("-" * 100)

display(
    player_performance[
        "classification"
    ]
    .value_counts(dropna=False)
    .rename_axis("classification")
    .reset_index(name="metricas")
)

# ---------------------------------------------------------------------
# Validación visual · Rubén Duarte, Apertura
# ---------------------------------------------------------------------

print("\n" + "=" * 100)
print("RUBÉN DUARTE · APERTURA 2025/2026")
print("=" * 100)

ruben = player_performance[
    (player_performance["player_id"] == 37)
    & (player_performance["tournament_name"] == "Apertura")
].copy()

print(
    f"\nPPS: {ruben['pps_v1'].iloc[0]:.2f}"
    f" | Ranking posición: {int(ruben['pps_position_rank'].iloc[0])}"
    f" | IGR: {ruben['igr_v1'].iloc[0]:.2f}"
    f" | Ranking IGR: {int(ruben['igr_rank'].iloc[0])}"
    f" | Confiabilidad: {ruben['igr_reliability'].iloc[0] * 100:.1f}%"
)

display(
    ruben[
        [
            "metric_label",
            "player_value",
            "reference_value",
            "difference_pct",
            "classification",
            "benchmark_n",
        ]
    ].round(
        {
            "player_value": 3,
            "reference_value": 3,
            "difference_pct": 1,
        }
    )
)

RENDIMIENTO INDIVIDUAL · PPS / IGR + MÉTRICAS EXPLICATIVAS

Jugador-torneo esperados:       39
Jugador-torneo integrados:      39
Métricas explicativas:          231
Duplicados jugador-métrica:     0
Filas sin métrica:             0

CLASIFICACIONES
----------------------------------------------------------------------------------------------------


,classification,metricas
0,Por debajo,91
1,Destaca,79
2,En línea,61



RUBÉN DUARTE · APERTURA 2025/2026

PPS: 62.34 | Ranking posición: 9 | IGR: 78.30 | Ranking IGR: 1 | Confiabilidad: 73.7%


,metric_label,player_value,reference_value,difference_pct,classification,benchmark_n
0,Entradas,1.643,1.576,4.2,En línea,69
1,Intercepciones,1.071,0.832,28.8,Destaca,69
2,Recuperaciones,4.000,3.084,29.7,Destaca,69
3,Despejes,4.571,3.980,14.8,Destaca,69
4,Duelos ganados,4.214,3.978,5.9,En línea,69
5,Precisión de pase,0.877,0.852,2.9,En línea,69


In [16]:
# ---------------------------------------------------------------------
# Paso 47 · Exportación de métricas explicativas del perfil individual
# ---------------------------------------------------------------------

from pathlib import Path

output_path = Path(
    "../data/processed/pumas_player_profile_2025_2026.csv"
)

profile_output_cols = [
    "season_name",
    "tournament_name",
    "player_id",
    "player_name",
    "position_group",
    "metric",
    "metric_label",
    "direction",
    "display_format",
    "player_value",
    "reference_value",
    "absolute_difference",
    "relative_difference",
    "difference_pct",
    "classification",
    "benchmark_n",
]

profile_output = player_performance[
    profile_output_cols
].copy()

# Orden estable para facilitar inspección y consumo posterior.
profile_output = profile_output.sort_values(
    [
        "tournament_name",
        "player_name",
        "metric",
    ]
).reset_index(drop=True)

# ---------------------------------------------------------------------
# Validaciones antes de exportar
# ---------------------------------------------------------------------

assert len(profile_output) == 231

assert (
    profile_output[
        ["season_name", "tournament_name", "player_id", "metric"]
    ]
    .duplicated()
    .sum()
    == 0
)

assert profile_output["metric"].notna().all()
assert profile_output["reference_value"].notna().all()
assert profile_output["classification"].notna().all()

assert set(profile_output["classification"].unique()) == {
    "Destaca",
    "En línea",
    "Por debajo",
}

# ---------------------------------------------------------------------
# Exportación
# ---------------------------------------------------------------------

profile_output.to_csv(
    output_path,
    index=False,
)

print("=" * 100)
print("EXPORTACIÓN · PERFIL INDIVIDUAL")
print("=" * 100)

print(f"\nArchivo: {output_path}")
print(f"Filas:   {len(profile_output)}")
print(f"Columnas: {len(profile_output.columns)}")

print("\nColumnas exportadas:")
for column in profile_output.columns:
    print(f"  - {column}")

print("\nClasificaciones:")
print(
    profile_output["classification"]
    .value_counts()
    .to_string()
)

print("\nVALIDACIÓN: OK")

EXPORTACIÓN · PERFIL INDIVIDUAL

Archivo: ../data/processed/pumas_player_profile_2025_2026.csv
Filas:   231
Columnas: 16

Columnas exportadas:
  - season_name
  - tournament_name
  - player_id
  - player_name
  - position_group
  - metric
  - metric_label
  - direction
  - display_format
  - player_value
  - reference_value
  - absolute_difference
  - relative_difference
  - difference_pct
  - classification
  - benchmark_n

Clasificaciones:
classification
Por debajo    91
Destaca       79
En línea      61

VALIDACIÓN: OK


In [17]:
# ---------------------------------------------------------------------
# Paso 48 · Validación final del artefacto exportado
# ---------------------------------------------------------------------

saved_profile = pd.read_csv(
    "../data/processed/pumas_player_profile_2025_2026.csv"
)

print("=" * 100)
print("VALIDACIÓN FINAL · pumas_player_profile_2025_2026.csv")
print("=" * 100)

print(f"\nShape: {saved_profile.shape}")

print(
    "Jugador-torneo: "
    f"{saved_profile[['season_name', 'tournament_name', 'player_id']].drop_duplicates().shape[0]}"
)

print(
    "Duplicados jugador-métrica: "
    f"{saved_profile.duplicated(
        subset=['season_name', 'tournament_name', 'player_id', 'metric']
    ).sum()}"
)

print(f"Valores nulos totales: {saved_profile.isna().sum().sum()}")

print("\nCobertura por torneo:")
display(
    saved_profile.groupby("tournament_name").agg(
        jugadores=("player_id", "nunique"),
        metricas=("metric", "size"),
    )
)

print("\nCobertura por posición:")
display(
    saved_profile.groupby("position_group").agg(
        jugadores_torneo=("player_id", "count"),
        metricas=("metric", "size"),
    )
)

print("\nRubén Duarte · Apertura:")
display(
    saved_profile[
        (saved_profile["player_id"] == 37)
        & (saved_profile["tournament_name"] == "Apertura")
    ][
        [
            "metric_label",
            "player_value",
            "reference_value",
            "difference_pct",
            "classification",
        ]
    ]
)

print("\nVALIDACIÓN DESDE DISCO: OK")

VALIDACIÓN FINAL · pumas_player_profile_2025_2026.csv

Shape: (231, 16)
Jugador-torneo: 39
Duplicados jugador-métrica: 0
Valores nulos totales: 0

Cobertura por torneo:


,jugadores,metricas
tournament_name,,
Apertura,19,113
Clausura,20,118



Cobertura por posición:


,jugadores_torneo,metricas
position_group,,
DF,90,90
FW,72,72
GK,15,15
MF,54,54



Rubén Duarte · Apertura:


,metric_label,player_value,reference_value,difference_pct,classification
83,Recuperaciones,4.000000,3.084025,29.700637,Destaca
84,Despejes,4.571429,3.980433,14.847511,Destaca
85,Duelos ganados,4.214286,3.978184,5.934902,En línea
86,Intercepciones,1.071429,0.831792,28.809604,Destaca
87,Precisión de pase,0.876614,0.851670,2.928815,En línea
88,Entradas,1.642857,1.576458,4.211921,En línea



VALIDACIÓN DESDE DISCO: OK
